# 1. Introduction

Part of the `nuclearff` [User Tutorial](https://nolmacdonald.github.io/nuclearff/tutorial/index.html) notebook series -- every notebook shares one running `./demo` directory and depends on the ones before it having already run. See [README.md](README.md) for the full run order.

`nuclearff` is a Python package for fantasy football research and valuation. It combines two data sources -- real NFL statistics from the [nflverse](https://github.com/nflverse) ecosystem (via [nflreadpy](https://github.com/nflverse/nflreadpy)) and league, roster, and matchup data from the [Sleeper API](https://docs.sleeper.com) -- into a single toolkit for scoring players under your league's own rules, projecting future performance, valuing a draft, simulating outcomes, and mining your league's own history.

This notebook series walks through every part of that toolkit as a Python library, chapter by chapter, in the order you would actually reach for it: connect to a league, capture and store its data, score and project players, value a draft, and finally dig into your league's own history and trades. Every example runs against a real Sleeper league (`1367225133634191360`, `NUCLEARFF REDRAFT`) with real output -- your own league's numbers will differ, but the shape of the output won't. Mirrors the [Introduction](https://nolmacdonald.github.io/nuclearff/tutorial/01_introduction.html) docs page.

Notebooks `01` through `17` are plain Python -- importing functions and classes and calling them directly, the way you would inside a notebook or a script. `nuclearff` also ships a command-line interface that wraps most of this into repeatable, on-disk commands; that's covered on its own, at the end, in `18_cli_reference.ipynb`.

> **Why this matters:** the docs page and this notebook cover the same ground twice on purpose. Reading the *why* in prose (the docs) and then typing the *how* yourself, cell by cell, with real output appearing as you go, builds a different kind of understanding than either alone -- this is the same reason the original 3-notebook set (#89) existed, just now kept in sync with all 18 chapters instead of 3 old, now-retired doc pages (#178).

## Installation

`nuclearff` is not published to PyPI. Clone the repository and install it with `uv`:

```bash
git clone https://github.com/nolmacdonald/nuclearff.git
cd nuclearff
uv sync --frozen
```

`--frozen` installs exactly what `uv.lock` specifies. Install the development extras to get `jupyter`, `pytest`, and the other tools used throughout this tutorial:

```bash
uv sync --frozen --extra dev
```

Every cell below assumes you're running inside that environment -- this notebook itself was executed with `uv run jupyter nbconvert --execute`, not a separately installed copy.

## Data Sources

**nflverse / nflreadpy** -- [nflreadpy](https://github.com/nflverse/nflreadpy) provides NFL play-by-play, player, roster, schedule, and advanced statistics data as `polars.DataFrame` objects. Nothing here requires an account or a key; data is cached locally between calls.

**Sleeper API** -- the [Sleeper API](https://docs.sleeper.com) is a read-only, unauthenticated HTTP API exposing leagues, rosters, matchups, transactions, and player metadata. Sleeper asks clients to stay under 1000 calls per minute -- `SleeperClient` (`03_sleeper_api.ipynb`) paces requests for you automatically.

## A First Look

Before touching Sleeper at all, here is `nuclearff` pulling real NFL statistics and configuring its own logging:

In [1]:
import logging

import nflreadpy as nfl

from nuclearff import configure_logging

configure_logging(level=logging.INFO)

# Weekly player statistics for the 2023 and 2024 seasons
player_stats = nfl.load_player_stats([2023, 2024])
player_stats.select(
    ["player_display_name", "season", "week", "position", "fantasy_points"]
).head(5)

player_display_name,season,week,position,fantasy_points
str,i32,i32,str,f64
"""Aaron Rodgers""",2023,1,"""QB""",0.0
"""Matt Prater""",2023,1,"""K""",0.0
"""Nick Folk""",2023,1,"""K""",0.0
"""Calais Campbell""",2023,1,"""DE""",0.0
"""Matthew Stafford""",2023,1,"""QB""",14.46


That real query returns tens of thousands of player-week rows across two seasons -- every position, including kickers and defensive linemen, since `load_player_stats` returns everyone the box score tracks. `fantasy_points` here is nflreadpy's own generic scoring, not your league's -- `05_scoring_engine.ipynb` covers replacing it with your league's actual rules.

In [2]:
player_stats.height

37626

`nflreadpy` returns [polars](https://docs.pola.rs) DataFrames, and so does everything in `nuclearff` that consumes them. Use `.to_pandas()` if a downstream tool expects a `pandas.DataFrame` instead.

## What's Next

`02_configuration.ipynb` covers `nuclearff`'s own configuration -- where data lives on disk, and the typed settings that drive scoring and projection -- before `03_sleeper_api.ipynb` connects to a real league.

## See Also

- [Introduction](https://nolmacdonald.github.io/nuclearff/tutorial/01_introduction.html) -- the matching docs page.
- [API Reference](https://nolmacdonald.github.io/nuclearff/api/index.html) -- full reference for every public class and function.